# 700 — Program Signature Construction

## Objective

Construct and validate the frozen Phase 7 program-query representations and the outcome-blind preparatory objects required by the prespecified conditional-null and benchmark designs before notebook 701 connectivity inference begins.

Notebook 700 is a preparatory and computational-calibration notebook. It does not generate perturbagen rankings, observed program–perturbagen connectivity results, conditional-randomization p-values, multiplicity results, compound prioritization, or mechanism-of-action summaries.

## Analytical boundary

This notebook consumes the registered frozen Phase 4, Phase 3, and notebook-110 handoffs as authoritative inputs.

It may use the frozen LINCS Level 5 matrix only for prospectively authorized outcome-blind quantities, including gene-level perturbational dispersion and the namespace-3 computational benchmark. It must not apply the real frozen program queries to the observed perturbational matrix in a way that computes or exposes real program–perturbagen connectivity.

The notebook does not:

* alter frozen Phase 4 program membership, weights, sign, or identity;
* use Phase 5 or Phase 6 evidence for query construction, eligibility, prioritization, or rescue;
* inspect compound connectivity or rank perturbagens;
* execute the primary inference reserved for notebook 701; or
* select the final Monte Carlo `Bmax` using observed scientific results.

The resistance-like orientation used by Phase 7 is derived only from frozen upstream Phase 3/4 evidence under the governing Phase 7 contract. It is stored separately from the unchanged Phase 4 gene weights.


In [30]:
# =============================================================================
# Imports
# =============================================================================

import h5py
import numpy as np
import pandas as pd

import shutil
import tempfile
from pathlib import Path

import json
import os
import platform
import subprocess
import sys

from threadpoolctl import threadpool_info

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Resolve frozen Phase 7 preparation inputs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_transcriptomic_gene_weights",
    "phase3.311.program_robustness_summary",
    "phase1.110.raw_input_inventory",
    "phase1.110.program_representation_fidelity",
    "phase1.110.bing_gene_handoff",
    "phase1.110.landmark_gene_handoff",
    "phase1.110.primary_signature_handoff",
    "phase1.110.primary_perturbagen_coverage",
)

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

for artifact_id, path in input_paths.items():
    print(
        f"{artifact_id}: "
        f"{project_relative_path(path)}"
    )


phase4.401.consensus_transcriptomic_program_catalog: data/processed/consensus_programs/401_consensus_transcriptomic_program_catalog.csv
phase4.401.consensus_transcriptomic_gene_weights: data/processed/consensus_programs/401_consensus_transcriptomic_gene_weights.csv
phase3.311.program_robustness_summary: data/processed/cellline_programs/311_program_robustness_summary.csv
phase1.110.raw_input_inventory: data/interim/perturbational/110_lincs_raw_input_inventory.csv
phase1.110.program_representation_fidelity: data/interim/perturbational/110_lincs_program_representation_fidelity.csv
phase1.110.bing_gene_handoff: data/interim/perturbational/110_lincs_bing_consensus_gene_handoff.csv
phase1.110.landmark_gene_handoff: data/interim/perturbational/110_lincs_landmark_consensus_gene_handoff.csv
phase1.110.primary_signature_handoff: data/interim/perturbational/110_lincs_primary_signature_handoff.parquet
phase1.110.primary_perturbagen_coverage: data/interim/perturbational/110_lincs_primary_perturbage

In [3]:
# =============================================================================
# Frozen Phase 7 program order
# =============================================================================

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

In [4]:
# =============================================================================
# Load frozen Phase 7 preparation inputs
# =============================================================================

program_catalog = pd.read_csv(
    input_paths[
        "phase4.401.consensus_transcriptomic_program_catalog"
    ]
)

phase4_gene_weights = pd.read_csv(
    input_paths[
        "phase4.401.consensus_transcriptomic_gene_weights"
    ]
)

phase3_robustness = pd.read_csv(
    input_paths[
        "phase3.311.program_robustness_summary"
    ]
)

raw_input_inventory = pd.read_csv(
    input_paths[
        "phase1.110.raw_input_inventory"
    ]
)

representation_fidelity = pd.read_csv(
    input_paths[
        "phase1.110.program_representation_fidelity"
    ]
)

bing_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.bing_gene_handoff"
    ]
)

landmark_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.landmark_gene_handoff"
    ]
)

primary_signature_handoff = pd.read_parquet(
    input_paths[
        "phase1.110.primary_signature_handoff"
    ]
)

primary_perturbagen_coverage = pd.read_csv(
    input_paths[
        "phase1.110.primary_perturbagen_coverage"
    ]
)

print("Program catalog shape:", program_catalog.shape)
print("Phase 4 gene weights shape:", phase4_gene_weights.shape)
print("Phase 3 robustness shape:", phase3_robustness.shape)
print("Raw input inventory shape:", raw_input_inventory.shape)
print("Representation fidelity shape:", representation_fidelity.shape)
print("BING handoff shape:", bing_gene_handoff.shape)
print("Landmark handoff shape:", landmark_gene_handoff.shape)
print(
    "Primary signature handoff shape:",
    primary_signature_handoff.shape,
)
print(
    "Primary perturbagen coverage shape:",
    primary_perturbagen_coverage.shape,
)

Program catalog shape: (3, 21)
Phase 4 gene weights shape: (7167, 8)
Phase 3 robustness shape: (10, 33)
Raw input inventory shape: (8, 4)
Representation fidelity shape: (3, 19)
BING handoff shape: (4485, 14)
Landmark handoff shape: (270, 14)
Primary signature handoff shape: (362036, 20)
Primary perturbagen coverage shape: (7914, 4)


In [5]:
# =============================================================================
# Derive resistance-like orientation from frozen Phase 3/4 evidence
# =============================================================================

program_source_metadata = (
    program_catalog[
        [
            "consensus_program_id",
            "cell_line_program",
            "orientation_multiplier",
        ]
    ]
    .rename(
        columns={
            "consensus_program_id": "program_id",
            "cell_line_program": "source_cell_line_program",
            "orientation_multiplier": (
                "phase4_orientation_multiplier"
            ),
        }
    )
    .merge(
        phase3_robustness[
            [
                "program_id",
                "lineage_adjusted_rho",
                "robustness_category",
            ]
        ].rename(
            columns={
                "program_id": "source_cell_line_program",
                "lineage_adjusted_rho": (
                    "phase3_lineage_adjusted_rho"
                ),
                "robustness_category": (
                    "phase3_robustness_category"
                ),
            }
        ),
        on="source_cell_line_program",
        how="left",
        validate="one_to_one",
    )
    .sort_values("program_id")
    .reset_index(drop=True)
)

if program_source_metadata[
    [
        "phase4_orientation_multiplier",
        "phase3_lineage_adjusted_rho",
    ]
].isna().any().any():
    raise RuntimeError(
        "Incomplete frozen Phase 3/4 orientation metadata."
    )

program_source_metadata[
    "resistance_like_orientation_multiplier"
] = np.sign(
    program_source_metadata[
        "phase4_orientation_multiplier"
    ]
    * program_source_metadata[
        "phase3_lineage_adjusted_rho"
    ]
).astype("int8")


In [6]:
# =============================================================================
# Validate the prospectively frozen resistance-like orientation
# =============================================================================

# This exact map is a contract cross-check, not an alternate orientation
# method. The multiplier itself is derived only from frozen upstream evidence.
EXPECTED_RESISTANCE_LIKE_MULTIPLIERS = {
    "CONSENSUS_TX_01": -1,
    "CONSENSUS_TX_02": 1,
    "CONSENSUS_TX_03": 1,
}

observed_multipliers = (
    program_source_metadata
    .set_index("program_id")[
        "resistance_like_orientation_multiplier"
    ]
    .astype(int)
    .to_dict()
)

if observed_multipliers != EXPECTED_RESISTANCE_LIKE_MULTIPLIERS:
    raise RuntimeError(
        "Derived resistance-like orientation does not "
        "match the frozen Phase 7 contract."
    )

display(program_source_metadata)


,program_id,source_cell_line_program,phase4_orientation_multiplier,phase3_lineage_adjusted_rho,phase3_robustness_category,resistance_like_orientation_multiplier
0,CONSENSUS_TX_01,ICA_PROGRAM_09,1,-0.038606,CONTEXT_SENSITIVE_CANDIDATE,-1
1,CONSENSUS_TX_02,ICA_PROGRAM_29,-1,-0.097379,ROBUSTNESS_SUPPORTED_CANDIDATE,1
2,CONSENSUS_TX_03,ICA_PROGRAM_13,-1,-0.136528,ROBUSTNESS_SUPPORTED_CANDIDATE,1


In [7]:
# =============================================================================
# Materialize deterministic continuous BING query definitions
# =============================================================================

program_query_definitions = (
    bing_gene_handoff[
        [
            "consensus_program_id",
            "gene_id",
            "gene_symbol",
            "feature_space",
            "gctx_gene_index",
            "consensus_weight",
        ]
    ]
    .rename(
        columns={
            "consensus_program_id": "program_id",
            "consensus_weight": "frozen_weight",
        }
    )
    .merge(
        program_source_metadata,
        on="program_id",
        how="left",
        validate="many_to_one",
    )
    .assign(
        gene_id=lambda data: pd.to_numeric(
            data["gene_id"],
            errors="raise",
        ).astype("int64"),
        gctx_gene_index=lambda data: pd.to_numeric(
            data["gctx_gene_index"],
            errors="raise",
        ).astype("int64"),
        primary_bing_member=True,
        landmark_sensitivity_member=lambda data: (
            data["feature_space"].eq("landmark")
        ),
    )
    .sort_values(
        [
            "program_id",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

program_query_definitions["query_gene_order"] = (
    program_query_definitions
    .groupby(
        "program_id",
        sort=False,
    )
    .cumcount()
    .astype("int64")
)


In [8]:
# =============================================================================
# Validate continuous BING query definitions
# =============================================================================

frozen_weight_values = (
    program_query_definitions["frozen_weight"]
    .to_numpy(dtype=np.float64)
)

if not np.isfinite(frozen_weight_values).all():
    raise RuntimeError(
        "Program query contains a non-finite frozen weight."
    )

query_norms = (
    program_query_definitions
    .groupby(
        "program_id",
        sort=False,
    )["frozen_weight"]
    .apply(
        lambda values: np.linalg.norm(
            values.to_numpy(dtype=np.float64)
        )
    )
    .reindex(PROGRAM_IDS)
)

if not query_norms.gt(0).all():
    raise RuntimeError(
        f"Program query has zero norm: "
        f"{query_norms.to_dict()}"
    )

if len(program_query_definitions) != 4485:
    raise RuntimeError(
        "Unexpected total number of BING "
        "query-definition rows."
    )

if program_query_definitions[
    [
        "program_id",
        "gene_id",
    ]
].duplicated().any():
    raise RuntimeError(
        "Duplicate program × gene rows in query definitions."
    )

query_summary = (
    program_query_definitions
    .groupby(
        "program_id",
        as_index=False,
    )
    .agg(
        bing_genes=("gene_id", "nunique"),
        landmark_genes=(
            "landmark_sensitivity_member",
            "sum",
        ),
        positive_weights=(
            "frozen_weight",
            lambda values: int((values > 0).sum()),
        ),
        negative_weights=(
            "frozen_weight",
            lambda values: int((values < 0).sum()),
        ),
    )
)

query_summary["l2_norm"] = (
    query_summary["program_id"].map(query_norms)
)

display(query_summary)


,program_id,bing_genes,landmark_genes,positive_weights,negative_weights,l2_norm
0,CONSENSUS_TX_01,1495,90,510,985,0.850741
1,CONSENSUS_TX_02,1495,90,864,631,0.801657
2,CONSENSUS_TX_03,1495,90,712,783,0.813026


In [9]:
# =============================================================================
# Select fixed CMap-style 150 / 150 query membership
# =============================================================================

program_query_definitions["cmap_style_arm"] = pd.Series(
    pd.NA,
    index=program_query_definitions.index,
    dtype="string",
)

for program_id in PROGRAM_IDS:
    program_mask = (
        program_query_definitions[
            "program_id"
        ].eq(program_id)
    )

    positive_candidates = (
        program_query_definitions.loc[
            program_mask
            & program_query_definitions[
                "frozen_weight"
            ].gt(0)
        ]
        .sort_values(
            [
                "frozen_weight",
                "gene_id",
            ],
            ascending=[False, True],
            kind="mergesort",
        )
    )

    negative_candidates = (
        program_query_definitions.loc[
            program_mask
            & program_query_definitions[
                "frozen_weight"
            ].lt(0)
        ]
        .sort_values(
            [
                "frozen_weight",
                "gene_id",
            ],
            ascending=[True, True],
            kind="mergesort",
        )
    )

    if len(positive_candidates) < 150:
        raise RuntimeError(
            f"{program_id} has fewer than "
            "150 positive BING weights."
        )

    if len(negative_candidates) < 150:
        raise RuntimeError(
            f"{program_id} has fewer than "
            "150 negative BING weights."
        )

    program_query_definitions.loc[
        positive_candidates.head(150).index,
        "cmap_style_arm",
    ] = "up"

    program_query_definitions.loc[
        negative_candidates.head(150).index,
        "cmap_style_arm",
    ] = "down"

program_query_definitions["cmap_style_member"] = (
    program_query_definitions[
        "cmap_style_arm"
    ].notna()
)


In [10]:
# =============================================================================
# Validate fixed CMap-style query membership
# =============================================================================

cmap_membership_summary = (
    program_query_definitions.loc[
        program_query_definitions[
            "cmap_style_member"
        ]
    ]
    .groupby(
        [
            "program_id",
            "cmap_style_arm",
        ],
        as_index=False,
    )
    .agg(
        genes=("gene_id", "nunique"),
    )
    .sort_values(
        [
            "program_id",
            "cmap_style_arm",
        ]
    )
    .reset_index(drop=True)
)

cmap_count_matrix = (
    cmap_membership_summary
    .pivot(
        index="program_id",
        columns="cmap_style_arm",
        values="genes",
    )
    .reindex(PROGRAM_IDS)
)

if not (
    cmap_count_matrix["up"].eq(150).all()
    and cmap_count_matrix["down"].eq(150).all()
):
    raise RuntimeError(
        "Fixed CMap-style query membership is not "
        "150 / 150 for every frozen program."
    )

display(cmap_membership_summary)


,program_id,cmap_style_arm,genes
0,CONSENSUS_TX_01,down,150
1,CONSENSUS_TX_01,up,150
2,CONSENSUS_TX_02,down,150
3,CONSENSUS_TX_02,up,150
4,CONSENSUS_TX_03,down,150
5,CONSENSUS_TX_03,up,150


In [11]:
# =============================================================================
# Resolve raw LINCS resources required for MAD stratification
# =============================================================================

raw_resource_paths = (
    raw_input_inventory
    .set_index("resource")["relative_path"]
)

LINCS_GENEINFO_PATH = (
    Paths.root
    / raw_resource_paths.loc["geneinfo"]
)

LINCS_LEVEL5_TRT_CP_PATH = (
    Paths.root
    / raw_resource_paths.loc["level5_trt_cp"]
)

for resource_name, path in {
    "geneinfo": LINCS_GENEINFO_PATH,
    "level5_trt_cp": LINCS_LEVEL5_TRT_CP_PATH,
}.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required LINCS resource not found: {path}"
        )

    print(
        f"{resource_name}: "
        f"{project_relative_path(path)}"
    )

geneinfo: data/raw/lincs/geneinfo_beta.txt
level5_trt_cp: data/raw/lincs/level5_beta_trt_cp_n720216x12328.gctx


In [12]:
# =============================================================================
# Construct the complete BING gene universe for MAD stratification
# =============================================================================

lincs_geneinfo = pd.read_csv(
    LINCS_GENEINFO_PATH,
    sep="\t",
    low_memory=False,
)

bing_gene_universe = (
    lincs_geneinfo.loc[
        lincs_geneinfo["feature_space"].isin(
            [
                "landmark",
                "best inferred",
            ]
        ),
        [
            "gene_id",
            "gene_symbol",
            "feature_space",
        ],
    ]
    .assign(
        gene_id=lambda data: pd.to_numeric(
            data["gene_id"],
            errors="raise",
        ).astype("int64")
    )
)

with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    gctx_gene_ids = [
        (
            value.decode("utf-8")
            if isinstance(value, bytes)
            else str(value)
        )
        for value in gctx["0/META/ROW/id"][:]
    ]

gctx_gene_index = pd.DataFrame(
    {
        "gene_id": pd.to_numeric(
            pd.Series(gctx_gene_ids),
            errors="raise",
        ).astype("int64"),
        "gctx_gene_index": np.arange(
            len(gctx_gene_ids),
            dtype=np.int64,
        ),
    }
)

bing_gene_universe = (
    bing_gene_universe
    .merge(
        gctx_gene_index,
        on="gene_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "gene_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if bing_gene_universe["gctx_gene_index"].isna().any():
    raise RuntimeError(
        "At least one BING gene is missing a GCTX gene index."
    )

if len(bing_gene_universe) != 10174:
    raise RuntimeError(
        "Unexpected complete BING gene-universe size: "
        f"{len(bing_gene_universe):,}"
    )

print(
    "Complete BING gene universe:",
    f"{len(bing_gene_universe):,}",
)

display(
    bing_gene_universe
    .groupby(
        "feature_space",
        as_index=False,
    )
    .agg(
        genes=("gene_id", "size"),
    )
)

Complete BING gene universe: 10,174


,feature_space,genes
0,best inferred,9196
1,landmark,978


In [14]:
# =============================================================================
# Prepare frozen primary-signature GCTX indices
# =============================================================================

primary_signature_indices = (
    pd.to_numeric(
        primary_signature_handoff["gctx_signature_index"],
        errors="raise",
    )
    .astype("int64")
    .to_numpy()
)

if len(primary_signature_indices) != 362036:
    raise RuntimeError(
        "Unexpected primary-signature count: "
        f"{len(primary_signature_indices):,}"
    )

if len(np.unique(primary_signature_indices)) != len(
    primary_signature_indices
):
    raise RuntimeError(
        "Primary-signature GCTX indices are not unique."
    )

primary_signature_indices = np.sort(
    primary_signature_indices,
    kind="stable",
)

print(
    "Frozen primary GCTX signature indices:",
    f"{len(primary_signature_indices):,}",
)
print(
    "Index range:",
    f"{primary_signature_indices[0]:,}"
    " to "
    f"{primary_signature_indices[-1]:,}",
)

Frozen primary GCTX signature indices: 362,036
Index range: 0 to 719,985


In [15]:
# =============================================================================
# Inspect GCTX storage layout for exact MAD computation
# =============================================================================

with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    matrix = gctx["0/DATA/0/matrix"]

    print("Matrix shape:", matrix.shape)
    print("Matrix dtype:", matrix.dtype)
    print("HDF5 chunks:", matrix.chunks)
    print("Compression:", matrix.compression)
    print(
        "Primary signatures:",
        f"{len(primary_signature_indices):,}",
    )

Matrix shape: (720216, 12328)
Matrix dtype: float32
HDF5 chunks: None
Compression: None
Primary signatures: 362,036


In [ ]:
# =============================================================================
# Check temporary-storage requirements for exact BING MAD computation
# =============================================================================

TEMP_DIR = Path(tempfile.gettempdir())

n_primary_signatures = len(primary_signature_indices)
n_bing_genes = len(bing_gene_universe)

bing_matrix_bytes = (
    n_primary_signatures
    * n_bing_genes
    * np.dtype(np.float32).itemsize
)

disk_usage = shutil.disk_usage(TEMP_DIR)

print("Temporary directory:", TEMP_DIR)
print(
    "Projected BING matrix size:",
    f"{bing_matrix_bytes / 1024**3:.2f} GiB",
)
print(
    "Free temporary-storage space:",
    f"{disk_usage.free / 1024**3:.2f} GiB",
)

Temporary directory: C:\Users\paula\AppData\Local\Temp
Projected BING matrix size: 13.72 GiB
Free temporary-storage space: 309.23 GiB


In [17]:
# =============================================================================
# Materialize primary-signature BING matrix for exact MAD computation
# =============================================================================

TEMP_BING_MATRIX_PATH = (
    TEMP_DIR
    / "phase7_700_primary_bing_matrix.npy"
)

READ_BLOCK_ROWS = 4096

bing_gctx_indices = (
    bing_gene_universe["gctx_gene_index"]
    .to_numpy(dtype=np.int64)
)

primary_bing_matrix = np.lib.format.open_memmap(
    TEMP_BING_MATRIX_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(
        n_bing_genes,
        n_primary_signatures,
    ),
)

with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    matrix = gctx["0/DATA/0/matrix"]

    for block_start in range(
        0,
        matrix.shape[0],
        READ_BLOCK_ROWS,
    ):
        block_stop = min(
            block_start + READ_BLOCK_ROWS,
            matrix.shape[0],
        )

        left = np.searchsorted(
            primary_signature_indices,
            block_start,
            side="left",
        )
        right = np.searchsorted(
            primary_signature_indices,
            block_stop,
            side="left",
        )

        if left == right:
            continue

        block = matrix[
            block_start:block_stop,
            :,
        ]

        local_primary_rows = (
            primary_signature_indices[left:right]
            - block_start
        )

        primary_bing_matrix[
            :,
            left:right,
        ] = block[
            local_primary_rows
        ][
            :,
            bing_gctx_indices
        ].T

primary_bing_matrix.flush()

print(
    "Temporary primary BING matrix:",
    project_relative_path(TEMP_BING_MATRIX_PATH)
    if TEMP_BING_MATRIX_PATH.is_relative_to(Paths.root)
    else TEMP_BING_MATRIX_PATH,
)
print(
    "Shape:",
    primary_bing_matrix.shape,
)
print(
    "Dtype:",
    primary_bing_matrix.dtype,
)
print(
    "Size:",
    f"{TEMP_BING_MATRIX_PATH.stat().st_size / 1024**3:.2f} GiB",
)

Temporary primary BING matrix: C:\Users\paula\AppData\Local\Temp\phase7_700_primary_bing_matrix.npy
Shape: (10174, 362036)
Dtype: float32
Size: 13.72 GiB


In [18]:
# =============================================================================
# Compute exact gene-level perturbational MAD across primary signatures
# =============================================================================

MAD_GENE_BATCH_SIZE = 32

gene_medians = np.empty(
    n_bing_genes,
    dtype=np.float64,
)

gene_mads = np.empty(
    n_bing_genes,
    dtype=np.float64,
)

for batch_start in range(
    0,
    n_bing_genes,
    MAD_GENE_BATCH_SIZE,
):
    batch_stop = min(
        batch_start + MAD_GENE_BATCH_SIZE,
        n_bing_genes,
    )

    values = np.asarray(
        primary_bing_matrix[
            batch_start:batch_stop,
            :
        ],
        dtype=np.float64,
    )

    if not np.isfinite(values).all():
        raise RuntimeError(
            "Non-finite Level 5 values encountered "
            f"for BING genes {batch_start}:{batch_stop}."
        )

    batch_medians = np.median(
        values,
        axis=1,
    )

    batch_mads = np.median(
        np.abs(
            values
            - batch_medians[:, np.newaxis]
        ),
        axis=1,
    )

    gene_medians[
        batch_start:batch_stop
    ] = batch_medians

    gene_mads[
        batch_start:batch_stop
    ] = batch_mads

bing_gene_dispersion = (
    bing_gene_universe
    .copy()
    .assign(
        perturbational_median=gene_medians,
        perturbational_mad=gene_mads,
    )
)

if not np.isfinite(
    bing_gene_dispersion[
        "perturbational_mad"
    ].to_numpy()
).all():
    raise RuntimeError(
        "At least one BING gene has a non-finite MAD."
    )

print(
    "BING genes with exact MAD:",
    f"{len(bing_gene_dispersion):,}",
)

display(
    bing_gene_dispersion[
        "perturbational_mad"
    ].describe()
)

BING genes with exact MAD: 10,174


count    10174.000000
mean         0.499179
std          0.019002
min          0.000000
25%          0.489956
50%          0.494616
75%          0.501806
max          0.612916
Name: perturbational_mad, dtype: float64

In [20]:
# =============================================================================
# Assign deterministic feature-space × MAD-decile strata
# =============================================================================

stratified_gene_blocks = []

for feature_space in (
    "best inferred",
    "landmark",
):
    feature_genes = (
        bing_gene_dispersion.loc[
            bing_gene_dispersion[
                "feature_space"
            ].eq(feature_space)
        ]
        .sort_values(
            [
                "perturbational_mad",
                "gene_id",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    n_genes = len(feature_genes)
    base_size, remainder = divmod(
        n_genes,
        10,
    )

    decile_sizes = np.array(
        [
            base_size + (decile < remainder)
            for decile in range(10)
        ],
        dtype=np.int64,
    )

    feature_genes["mad_decile"] = np.repeat(
        np.arange(
            1,
            11,
            dtype=np.int8,
        ),
        decile_sizes,
    )

    stratified_gene_blocks.append(
        feature_genes
    )

null_stratification_genes = (
    pd.concat(
        stratified_gene_blocks,
        ignore_index=True,
    )
)

stratum_order = (
    null_stratification_genes[
        [
            "feature_space",
            "mad_decile",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "feature_space",
            "mad_decile",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

stratum_order["stratum_id"] = np.arange(
    len(stratum_order),
    dtype=np.int64,
)

null_stratification_genes = (
    null_stratification_genes
    .merge(
        stratum_order,
        on=[
            "feature_space",
            "mad_decile",
        ],
        how="left",
        validate="many_to_one",
    )
)

stratum_summary = (
    null_stratification_genes
    .groupby(
        [
            "stratum_id",
            "feature_space",
            "mad_decile",
        ],
        as_index=False,
    )
    .agg(
        genes=("gene_id", "size"),
        min_mad=("perturbational_mad", "min"),
        max_mad=("perturbational_mad", "max"),
    )
)

print(
    "Null strata:",
    len(stratum_summary),
)
print(
    "Genes with MAD == 0:",
    int(
        null_stratification_genes[
            "perturbational_mad"
        ].eq(0).sum()
    ),
)

display(stratum_summary)

Null strata: 20
Genes with MAD == 0: 1


,stratum_id,feature_space,mad_decile,genes,min_mad,max_mad
0,0,best inferred,1,920,0.118600,0.486450
1,1,best inferred,2,920,0.486450,0.488641
2,2,best inferred,3,920,0.488645,0.490355
3,3,best inferred,4,920,0.490355,0.491942
4,4,best inferred,5,920,0.491942,0.493686
5,5,best inferred,6,920,0.493692,0.495577
6,6,best inferred,7,919,0.495578,0.497707
7,7,best inferred,8,919,0.497711,0.500791
8,8,best inferred,9,919,0.500794,0.505192
9,9,best inferred,10,919,0.505197,0.530183


In [22]:
# =============================================================================
# Map frozen program queries to the null-stratification strata
# =============================================================================

query_stratification = (
    program_query_definitions
    .merge(
        null_stratification_genes[
            [
                "gene_id",
                "feature_space",
                "perturbational_mad",
                "mad_decile",
                "stratum_id",
            ]
        ],
        on=[
            "gene_id",
            "feature_space",
        ],
        how="left",
        validate="many_to_one",
    )
)

if query_stratification[
    [
        "perturbational_mad",
        "mad_decile",
        "stratum_id",
    ]
].isna().any().any():
    raise RuntimeError(
        "At least one frozen query gene could not be assigned "
        "to a null stratum."
    )

query_counts = (
    query_stratification
    .groupby(
        [
            "program_id",
            "stratum_id",
        ],
        as_index=False,
    )
    .agg(
        query_genes=("gene_id", "size"),
    )
)

program_stratum_grid = (
    pd.DataFrame(
        {
            "program_id": PROGRAM_IDS,
        }
    )
    .merge(
        stratum_summary[
            [
                "stratum_id",
                "feature_space",
                "mad_decile",
                "genes",
            ]
        ],
        how="cross",
    )
    .rename(
        columns={
            "genes": "total_bing_genes",
        }
    )
)

query_stratum_summary = (
    program_stratum_grid
    .merge(
        query_counts,
        on=[
            "program_id",
            "stratum_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .fillna(
        {
            "query_genes": 0,
        }
    )
)

query_stratum_summary["query_genes"] = (
    query_stratum_summary["query_genes"]
    .astype("int64")
)

query_stratum_summary[
    "effectively_permutable_query_genes"
] = np.where(
    query_stratum_summary["query_genes"] > 1,
    query_stratum_summary["query_genes"],
    0,
)

query_stratum_summary[
    "degenerate_one_gene_stratum"
] = query_stratum_summary[
    "query_genes"
].eq(1)

print(
    "Program × stratum rows:",
    len(query_stratum_summary),
)
print(
    "Degenerate one-gene strata:",
    int(
        query_stratum_summary[
            "degenerate_one_gene_stratum"
        ].sum()
    ),
)
print(
    "Empty query strata:",
    int(
        query_stratum_summary[
            "query_genes"
        ].eq(0).sum()
    ),
)

display(query_stratum_summary)

Program × stratum rows: 60
Degenerate one-gene strata: 0
Empty query strata: 0


,program_id,stratum_id,feature_space,mad_decile,total_bing_genes,query_genes,effectively_permutable_query_genes,degenerate_one_gene_stratum
0,CONSENSUS_TX_01,0,best inferred,1,920,236,236,False
1,CONSENSUS_TX_01,1,best inferred,2,920,229,229,False
2,CONSENSUS_TX_01,2,best inferred,3,920,154,154,False
3,CONSENSUS_TX_01,3,best inferred,4,920,138,138,False
4,CONSENSUS_TX_01,4,best inferred,5,920,114,114,False
5,CONSENSUS_TX_01,5,best inferred,6,920,117,117,False
6,CONSENSUS_TX_01,6,best inferred,7,919,107,107,False
7,CONSENSUS_TX_01,7,best inferred,8,919,106,106,False
8,CONSENSUS_TX_01,8,best inferred,9,919,114,114,False
9,CONSENSUS_TX_01,9,best inferred,10,919,90,90,False


In [23]:
# =============================================================================
# Construct coupled three-program weight vectors for the primary null
# =============================================================================

coupled_query_weights = (
    query_stratification[
        [
            "gene_id",
            "gene_symbol",
            "feature_space",
            "perturbational_mad",
            "mad_decile",
            "stratum_id",
            "program_id",
            "frozen_weight",
        ]
    ]
    .pivot(
        index=[
            "gene_id",
            "gene_symbol",
            "feature_space",
            "perturbational_mad",
            "mad_decile",
            "stratum_id",
        ],
        columns="program_id",
        values="frozen_weight",
    )
    .reset_index()
)

coupled_query_weights.columns.name = None

if coupled_query_weights[
    list(PROGRAM_IDS)
].isna().any().any():
    raise RuntimeError(
        "At least one query gene lacks a complete "
        "three-program signed-weight vector."
    )

if len(coupled_query_weights) != 1495:
    raise RuntimeError(
        "Unexpected coupled-query gene count: "
        f"{len(coupled_query_weights):,}"
    )

coupled_query_weights = (
    coupled_query_weights
    .sort_values(
        [
            "stratum_id",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

coupled_query_weights["permutation_position"] = (
    coupled_query_weights
    .groupby(
        "stratum_id",
        sort=False,
    )
    .cumcount()
    .astype("int64")
)

positions_per_stratum = (
    coupled_query_weights
    .groupby(
        "stratum_id",
        as_index=False,
    )
    .agg(
        distinct_permutation_positions=(
            "permutation_position",
            "size",
        )
    )
)

query_stratum_summary = (
    query_stratum_summary
    .merge(
        positions_per_stratum,
        on="stratum_id",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Coupled query genes:",
    f"{len(coupled_query_weights):,}",
)
print(
    "Complete three-program weight vectors:",
    coupled_query_weights[
        list(PROGRAM_IDS)
    ].notna().all(axis=1).all(),
)

display(
    query_stratum_summary[
        [
            "program_id",
            "stratum_id",
            "feature_space",
            "mad_decile",
            "total_bing_genes",
            "query_genes",
            "effectively_permutable_query_genes",
            "degenerate_one_gene_stratum",
            "distinct_permutation_positions",
        ]
    ]
)

Coupled query genes: 1,495
Complete three-program weight vectors: True


,program_id,stratum_id,feature_space,mad_decile,total_bing_genes,query_genes,effectively_permutable_query_genes,degenerate_one_gene_stratum,distinct_permutation_positions
0,CONSENSUS_TX_01,0,best inferred,1,920,236,236,False,236
1,CONSENSUS_TX_01,1,best inferred,2,920,229,229,False,229
2,CONSENSUS_TX_01,2,best inferred,3,920,154,154,False,154
3,CONSENSUS_TX_01,3,best inferred,4,920,138,138,False,138
4,CONSENSUS_TX_01,4,best inferred,5,920,114,114,False,114
5,CONSENSUS_TX_01,5,best inferred,6,920,117,117,False,117
6,CONSENSUS_TX_01,6,best inferred,7,919,107,107,False,107
7,CONSENSUS_TX_01,7,best inferred,8,919,106,106,False,106
8,CONSENSUS_TX_01,8,best inferred,9,919,114,114,False,114
9,CONSENSUS_TX_01,9,best inferred,10,919,90,90,False,90


In [25]:
# =============================================================================
# Freeze deterministic Phase 7 RNG and permutation mapping
# =============================================================================

RNG_ENTROPY = 701

RNG_NAMESPACES = {
    "primary_conditional_null": 1,
    "competitive_null_sensitivity": 2,
    "outcome_blind_benchmark": 3,
}

FEATURE_SPACE_ORDER = (
    "best inferred",
    "landmark",
)

STRATUM_IDS = tuple(
    stratum_order["stratum_id"]
    .astype(int)
    .tolist()
)


def make_phase7_rng(
    namespace,
    replicate_id,
    stratum_id,
):
    seed_sequence = np.random.SeedSequence(
        entropy=RNG_ENTROPY,
        spawn_key=(
            int(namespace),
            int(replicate_id),
            int(stratum_id),
        ),
    )

    return np.random.Generator(
        np.random.PCG64DXSM(seed_sequence)
    )


def fisher_yates_permutation(
    n_positions,
    namespace,
    replicate_id,
    stratum_id,
):
    permutation = np.arange(
        n_positions,
        dtype=np.int64,
    )

    rng = make_phase7_rng(
        namespace=namespace,
        replicate_id=replicate_id,
        stratum_id=stratum_id,
    )

    for position in range(
        n_positions - 1,
        0,
        -1,
    ):
        swap_position = int(
            rng.integers(
                0,
                position + 1,
            )
        )

        permutation[
            position
        ], permutation[
            swap_position
        ] = (
            permutation[swap_position],
            permutation[position],
        )

    return permutation


print("RNG entropy:", RNG_ENTROPY)
print("RNG namespaces:", RNG_NAMESPACES)
print("Program order:", PROGRAM_IDS)
print("Feature-space order:", FEATURE_SPACE_ORDER)
print("Stratum order:", STRATUM_IDS)
print(
    "Permutation mapping:",
    "explicit Fisher-Yates, descending position order",
)

RNG entropy: 701
RNG namespaces: {'primary_conditional_null': 1, 'competitive_null_sensitivity': 2, 'outcome_blind_benchmark': 3}
Program order: ('CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03')
Feature-space order: ('best inferred', 'landmark')
Stratum order: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19)
Permutation mapping: explicit Fisher-Yates, descending position order


In [26]:
# =============================================================================
# Validate deterministic Phase 7 permutation streams
# =============================================================================

TEST_STRATUM_ID = 0
TEST_N_POSITIONS = int(
    positions_per_stratum.loc[
        positions_per_stratum[
            "stratum_id"
        ].eq(TEST_STRATUM_ID),
        "distinct_permutation_positions",
    ].iloc[0]
)

permutation_a = fisher_yates_permutation(
    n_positions=TEST_N_POSITIONS,
    namespace=RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
    replicate_id=0,
    stratum_id=TEST_STRATUM_ID,
)

permutation_b = fisher_yates_permutation(
    n_positions=TEST_N_POSITIONS,
    namespace=RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
    replicate_id=0,
    stratum_id=TEST_STRATUM_ID,
)

permutation_next_replicate = (
    fisher_yates_permutation(
        n_positions=TEST_N_POSITIONS,
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
        replicate_id=1,
        stratum_id=TEST_STRATUM_ID,
    )
)

permutation_next_stratum = (
    fisher_yates_permutation(
        n_positions=TEST_N_POSITIONS,
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
        replicate_id=0,
        stratum_id=1,
    )
)

expected_positions = np.arange(
    TEST_N_POSITIONS,
    dtype=np.int64,
)

if not np.array_equal(
    permutation_a,
    permutation_b,
):
    raise RuntimeError(
        "Identical Phase 7 RNG coordinates are not reproducible."
    )

if not np.array_equal(
    np.sort(permutation_a),
    expected_positions,
):
    raise RuntimeError(
        "Generated index vector is not a valid permutation."
    )

if np.array_equal(
    permutation_a,
    permutation_next_replicate,
):
    raise RuntimeError(
        "Adjacent replicate streams unexpectedly coincide."
    )

if np.array_equal(
    permutation_a,
    permutation_next_stratum,
):
    raise RuntimeError(
        "Adjacent stratum streams unexpectedly coincide."
    )

print("Deterministic RNG reproducibility: PASS")
print("Permutation validity: PASS")
print("Replicate-stream separation: PASS")
print("Stratum-stream separation: PASS")
print(
    "Test permutation positions:",
    TEST_N_POSITIONS,
)

Deterministic RNG reproducibility: PASS
Permutation validity: PASS
Replicate-stream separation: PASS
Stratum-stream separation: PASS
Test permutation positions: 236


In [27]:
# =============================================================================
# Materialize the stable null-stratification manifest in memory
# =============================================================================

query_permutation_map = (
    coupled_query_weights[
        [
            "gene_id",
            "stratum_id",
            "permutation_position",
        ]
    ]
    .rename(
        columns={
            "permutation_position": (
                "query_permutation_position"
            ),
        }
    )
)

null_stratification_manifest = (
    null_stratification_genes
    .sort_values(
        [
            "stratum_id",
            "perturbational_mad",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

null_stratification_manifest[
    "stratum_gene_order"
] = (
    null_stratification_manifest
    .groupby(
        "stratum_id",
        sort=False,
    )
    .cumcount()
    .astype("int64")
)

null_stratification_manifest = (
    null_stratification_manifest
    .merge(
        query_permutation_map,
        on=[
            "gene_id",
            "stratum_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

null_stratification_manifest[
    "primary_query_member"
] = null_stratification_manifest[
    "query_permutation_position"
].notna()

null_stratification_manifest[
    "query_permutation_position"
] = (
    null_stratification_manifest[
        "query_permutation_position"
    ]
    .astype("Int64")
)

if len(null_stratification_manifest) != 10174:
    raise RuntimeError(
        "Unexpected null-stratification manifest size."
    )

if not null_stratification_manifest[
    "gene_id"
].is_unique:
    raise RuntimeError(
        "Gene IDs are not unique in the "
        "null-stratification manifest."
    )

if (
    null_stratification_manifest[
        "stratum_id"
    ].nunique()
    != 20
):
    raise RuntimeError(
        "Unexpected number of null strata."
    )

if (
    null_stratification_manifest[
        "primary_query_member"
    ].sum()
    != 1495
):
    raise RuntimeError(
        "Unexpected primary-query membership "
        "in the null-stratification manifest."
    )

print(
    "Null-stratification manifest rows:",
    f"{len(null_stratification_manifest):,}",
)
print(
    "Frozen null strata:",
    null_stratification_manifest[
        "stratum_id"
    ].nunique(),
)
print(
    "Primary-query genes:",
    int(
        null_stratification_manifest[
            "primary_query_member"
        ].sum()
    ),
)

display(
    null_stratification_manifest.head()
)

Null-stratification manifest rows: 10,174
Frozen null strata: 20
Primary-query genes: 1495


,gene_id,gene_symbol,feature_space,gctx_gene_index,perturbational_median,perturbational_mad,mad_decile,stratum_id,stratum_gene_order,query_permutation_position,primary_query_member
0,7431,VIM,best inferred,9631,0.000000,0.118600,1,0,0,<NA>,False
1,3945,LDHB,best inferred,4878,0.000000,0.282002,1,0,1,<NA>,False
2,6277,S100A6,best inferred,8401,0.000000,0.348789,1,0,2,119,True
3,2171,FABP5,best inferred,2112,0.000000,0.356900,1,0,3,37,True
4,6171,RPL41,best inferred,8329,0.003999,0.405973,1,0,4,<NA>,False


In [29]:
# =============================================================================
# Validate the frozen null-stratification manifest
# =============================================================================

manifest_stratum_summary = (
    null_stratification_manifest
    .groupby(
        [
            "stratum_id",
            "feature_space",
            "mad_decile",
        ],
        as_index=False,
    )
    .agg(
        total_bing_genes=("gene_id", "size"),
        query_genes=("primary_query_member", "sum"),
    )
)

manifest_stratum_summary[
    "query_genes"
] = manifest_stratum_summary[
    "query_genes"
].astype("int64")

for stratum_id, stratum_data in (
    null_stratification_manifest.loc[
        null_stratification_manifest[
            "primary_query_member"
        ]
    ]
    .groupby(
        "stratum_id",
        sort=True,
    )
):
    observed_positions = (
        stratum_data[
            "query_permutation_position"
        ]
        .astype("int64")
        .sort_values()
        .to_numpy()
    )

    expected_positions = np.arange(
        len(stratum_data),
        dtype=np.int64,
    )

    if not np.array_equal(
        observed_positions,
        expected_positions,
    ):
        raise RuntimeError(
            "Non-contiguous query permutation positions "
            f"in stratum {stratum_id}."
        )

manifest_program_summary = (
    pd.DataFrame(
        {
            "program_id": PROGRAM_IDS,
        }
    )
    .merge(
        manifest_stratum_summary,
        how="cross",
    )
    .assign(
        effectively_permutable_query_genes=lambda data: (
            np.where(
                data["query_genes"] > 1,
                data["query_genes"],
                0,
            )
        ),
        degenerate_one_gene_stratum=lambda data: (
            data["query_genes"].eq(1)
        ),
        distinct_permutation_positions=lambda data: (
            data["query_genes"]
        ),
    )
    .sort_values(
        [
            "program_id",
            "stratum_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

comparison_columns = [
    "program_id",
    "stratum_id",
    "total_bing_genes",
    "query_genes",
    "effectively_permutable_query_genes",
    "degenerate_one_gene_stratum",
    "distinct_permutation_positions",
]

expected_summary = (
    query_stratum_summary[
        comparison_columns
    ]
    .sort_values(
        [
            "program_id",
            "stratum_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not manifest_program_summary[
    comparison_columns
].equals(expected_summary):
    raise RuntimeError(
        "Null-stratification manifest does not reproduce "
        "the frozen program × stratum summary."
    )

print("Null-stratification manifest validation: PASS")
print(
    "Program × stratum combinations:",
    len(manifest_program_summary),
)
print(
    "Degenerate one-gene strata:",
    int(
        manifest_program_summary[
            "degenerate_one_gene_stratum"
        ].sum()
    ),
)
print(
    "Minimum query genes per stratum:",
    int(
        manifest_program_summary[
            "query_genes"
        ].min()
    ),
)
print(
    "Maximum query genes per stratum:",
    int(
        manifest_program_summary[
            "query_genes"
        ].max()
    ),
)

Null-stratification manifest validation: PASS
Program × stratum combinations: 60
Degenerate one-gene strata: 0
Minimum query genes per stratum: 5
Maximum query genes per stratum: 236


In [31]:
# =============================================================================
# Inspect execution host before freezing the benchmark resource envelope
# =============================================================================

powershell_script = r"""
$cpu = Get-CimInstance Win32_Processor | Select-Object -First 1
$system = Get-CimInstance Win32_ComputerSystem

[PSCustomObject]@{
    cpu_model = $cpu.Name
    physical_cores = $cpu.NumberOfCores
    logical_cores = $cpu.NumberOfLogicalProcessors
    physical_ram_bytes = [int64]$system.TotalPhysicalMemory
} | ConvertTo-Json -Compress
"""

host_result = subprocess.run(
    [
        "powershell",
        "-NoProfile",
        "-Command",
        powershell_script,
    ],
    check=True,
    capture_output=True,
    text=True,
)

windows_hardware = json.loads(
    host_result.stdout.strip()
)

threading_info = threadpool_info()

execution_host_info = {
    "host_identifier": platform.node(),
    "operating_system": platform.platform(),
    "cpu_model": windows_hardware["cpu_model"].strip(),
    "physical_cores": int(
        windows_hardware["physical_cores"]
    ),
    "logical_cores": int(
        windows_hardware["logical_cores"]
    ),
    "physical_ram_gib": (
        int(windows_hardware["physical_ram_bytes"])
        / 1024**3
    ),
    "gpu_used": False,
    "gpu_model": None,
    "usable_gpu_vram_gib": None,
    "python_version": sys.version.split()[0],
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "h5py_version": h5py.__version__,
    "thread_environment": {
        variable: os.environ.get(variable)
        for variable in (
            "OMP_NUM_THREADS",
            "OPENBLAS_NUM_THREADS",
            "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS",
        )
    },
    "threadpool_info": threading_info,
}

print(
    "Host identifier:",
    execution_host_info["host_identifier"],
)
print(
    "CPU:",
    execution_host_info["cpu_model"],
)
print(
    "Physical / logical cores:",
    execution_host_info["physical_cores"],
    "/",
    execution_host_info["logical_cores"],
)
print(
    "Physical RAM:",
    f'{execution_host_info["physical_ram_gib"]:.2f} GiB',
)
print(
    "GPU used:",
    execution_host_info["gpu_used"],
)
print(
    "Python:",
    execution_host_info["python_version"],
)
print(
    "NumPy:",
    execution_host_info["numpy_version"],
)
print(
    "pandas:",
    execution_host_info["pandas_version"],
)
print(
    "h5py:",
    execution_host_info["h5py_version"],
)

print("\nThread environment:")
display(
    pd.DataFrame(
        execution_host_info[
            "thread_environment"
        ].items(),
        columns=[
            "variable",
            "value",
        ],
    )
)

print("Loaded numerical thread pools:")
display(
    pd.DataFrame(threading_info)
)

Host identifier: CompuPaulaCasa
CPU: Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz
Physical / logical cores: 4 / 8
Physical RAM: 19.80 GiB
GPU used: False
Python: 3.11.8
NumPy: 2.4.6
pandas: 3.0.3
h5py: 3.16.0

Thread environment:


,variable,value
0,OMP_NUM_THREADS,None
1,OPENBLAS_NUM_THREADS,None
2,MKL_NUM_THREADS,None
3,NUMEXPR_NUM_THREADS,None


Loaded numerical thread pools:


,user_api,internal_api,num_threads,prefix,filepath,version,threading_layer,architecture
0,blas,openblas,8,libscipy_openblas,C:\Users\paula\OneDrive\Documentos\Proyectos\p...,0.3.31.188.0,pthreads,SkylakeX


In [32]:
print(
    "Host identifier:",
    execution_host_info["host_identifier"],
)
print(
    "CPU:",
    execution_host_info["cpu_model"],
)
print(
    "Physical / logical cores:",
    execution_host_info["physical_cores"],
    "/",
    execution_host_info["logical_cores"],
)
print(
    "Physical RAM:",
    f'{execution_host_info["physical_ram_gib"]:.2f} GiB',
)
print(
    "GPU used:",
    execution_host_info["gpu_used"],
)
print(
    "Python:",
    execution_host_info["python_version"],
)
print(
    "NumPy:",
    execution_host_info["numpy_version"],
)
print(
    "pandas:",
    execution_host_info["pandas_version"],
)
print(
    "h5py:",
    execution_host_info["h5py_version"],
)

print("\nThread environment:")
print(
    pd.DataFrame(
        execution_host_info[
            "thread_environment"
        ].items(),
        columns=[
            "variable",
            "value",
        ],
    )
)

print("Loaded numerical thread pools:")
print(
    pd.DataFrame(threading_info)
)

Host identifier: CompuPaulaCasa
CPU: Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz
Physical / logical cores: 4 / 8
Physical RAM: 19.80 GiB
GPU used: False
Python: 3.11.8
NumPy: 2.4.6
pandas: 3.0.3
h5py: 3.16.0

Thread environment:
               variable value
0       OMP_NUM_THREADS  None
1  OPENBLAS_NUM_THREADS  None
2       MKL_NUM_THREADS  None
3   NUMEXPR_NUM_THREADS  None
Loaded numerical thread pools:
  user_api internal_api  num_threads             prefix  \
0     blas     openblas            8  libscipy_openblas   

                                            filepath       version  \
0  C:\Users\paula\OneDrive\Documentos\Proyectos\p...  0.3.31.188.0   

  threading_layer architecture  
0        pthreads     SkylakeX  


In [33]:
# =============================================================================
# Freeze and persist the pre-benchmark resource envelope
# =============================================================================

pip_freeze = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()

BENCHMARK_RESOURCE_ENVELOPE_PATH = (
    Paths.perturbational_hypotheses
    / "700_benchmark_resource_envelope.json"
)

BENCHMARK_RESOURCE_ENVELOPE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

benchmark_resource_envelope = {
    "execution_host": {
        "host_identifier": execution_host_info[
            "host_identifier"
        ],
        "operating_system": execution_host_info[
            "operating_system"
        ],
        "cpu_model": execution_host_info[
            "cpu_model"
        ],
        "physical_cores": execution_host_info[
            "physical_cores"
        ],
        "logical_cores": execution_host_info[
            "logical_cores"
        ],
        "physical_ram_gib": round(
            execution_host_info[
                "physical_ram_gib"
            ],
            6,
        ),
        "gpu_used": False,
        "gpu_model": None,
        "usable_gpu_vram_gib": None,
    },
    "resource_limits": {
        "maximum_ram_gib": 12.0,
        "maximum_projected_primary_wall_clock_hours": 24.0,
        "maximum_total_benchmark_wall_clock_hours": 2.0,
        "maximum_temporary_storage_gib": 64.0,
        "free_temporary_storage_gib_at_freeze": round(
            disk_usage.free / 1024**3,
            6,
        ),
    },
    "threading": {
        "environment": execution_host_info[
            "thread_environment"
        ],
        "threadpool_info": execution_host_info[
            "threadpool_info"
        ],
    },
    "checkpoint_restart_policy": {
        "checkpoint_interval_replicates": 1000,
        "checkpoint_at_case_completion": True,
        "checkpoint_write_mode": (
            "atomic temporary-file write followed by replace"
        ),
        "restart_rule": (
            "resume from the highest fully committed replicate "
            "boundary; discard any incomplete replicate"
        ),
        "rng_state_restoration_required": False,
        "rng_restart_basis": (
            "namespace × replicate_id × stratum_id"
        ),
    },
    "benchmark_design_freeze": {
        "rng_namespace": RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
        "candidate_bmax_ladder": [
            500000,
            250000,
            100000,
        ],
        "primary_hypothesis_count": 23742,
        "active_set_cases": [
            23742,
            11871,
            2375,
        ],
        "active_set_case_labels": [
            "maximum_100_percent",
            "reduced_50_percent",
            "reduced_10_percent_ceiling",
        ],
        "selection_rule": (
            "select the largest candidate Bmax whose "
            "conservative projected execution satisfies "
            "all frozen resource limits"
        ),
    },
    "software_environment": {
        "python": execution_host_info[
            "python_version"
        ],
        "numpy": execution_host_info[
            "numpy_version"
        ],
        "pandas": execution_host_info[
            "pandas_version"
        ],
        "h5py": execution_host_info[
            "h5py_version"
        ],
        "pip_freeze": pip_freeze,
    },
}

with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        benchmark_resource_envelope,
        handle,
        indent=2,
        sort_keys=True,
    )
    handle.write("\n")

if not BENCHMARK_RESOURCE_ENVELOPE_PATH.is_file():
    raise RuntimeError(
        "Benchmark resource envelope was not persisted."
    )

with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    persisted_resource_envelope = json.load(handle)

if (
    persisted_resource_envelope
    != benchmark_resource_envelope
):
    raise RuntimeError(
        "Persisted benchmark resource envelope "
        "does not match the frozen in-memory object."
    )

print("Pre-benchmark resource envelope: FROZEN")
print(
    "Path:",
    project_relative_path(
        BENCHMARK_RESOURCE_ENVELOPE_PATH
    ),
)
print(
    "Maximum RAM:",
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"],
    "GiB",
)
print(
    "Maximum projected primary runtime:",
    benchmark_resource_envelope[
        "resource_limits"
    ][
        "maximum_projected_primary_wall_clock_hours"
    ],
    "hours",
)
print(
    "Temporary-storage allowance:",
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_temporary_storage_gib"],
    "GiB",
)
print(
    "Checkpoint interval:",
    benchmark_resource_envelope[
        "checkpoint_restart_policy"
    ]["checkpoint_interval_replicates"],
    "replicates",
)
print(
    "Benchmark active-set cases:",
    benchmark_resource_envelope[
        "benchmark_design_freeze"
    ]["active_set_cases"],
)

Pre-benchmark resource envelope: FROZEN
Path: data/processed/perturbational_hypotheses/700_benchmark_resource_envelope.json
Maximum RAM: 12.0 GiB
Maximum projected primary runtime: 24.0 hours
Temporary-storage allowance: 64.0 GiB
Checkpoint interval: 1000 replicates
Benchmark active-set cases: [23742, 11871, 2375]


In [34]:
# =============================================================================
# Align primary-signature metadata and define the frozen aggregation hierarchy
# =============================================================================

primary_signature_metadata = (
    primary_signature_handoff
    .sort_values(
        "gctx_signature_index",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

aligned_signature_indices = (
    primary_signature_metadata[
        "gctx_signature_index"
    ]
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    aligned_signature_indices,
    primary_signature_indices,
):
    raise RuntimeError(
        "Primary-signature metadata are not aligned "
        "with the temporary BING matrix."
    )

aggregation_columns = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

if primary_signature_metadata[
    aggregation_columns
].isna().any().any():
    raise RuntimeError(
        "Frozen aggregation metadata contain missing values."
    )

condition_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

timepoint_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_time",
    "pert_time_unit",
]

cell_line_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
]

lineage_keys = [
    "pert_id",
    "cell_lineage",
]

hierarchy_counts = {
    "signatures": len(primary_signature_metadata),
    "conditions": (
        primary_signature_metadata[
            condition_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "timepoints_within_cell": (
        primary_signature_metadata[
            timepoint_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "perturbagen_cell_lines": (
        primary_signature_metadata[
            cell_line_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "perturbagen_lineages": (
        primary_signature_metadata[
            lineage_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "perturbagens": (
        primary_signature_metadata[
            "pert_id"
        ]
        .nunique()
    ),
}

if hierarchy_counts["perturbagens"] != 7914:
    raise RuntimeError(
        "Unexpected perturbagen count in the "
        "frozen aggregation hierarchy."
    )

print("Primary-signature metadata alignment: PASS")
print(
    "Frozen aggregation hierarchy:"
)
for level, count in hierarchy_counts.items():
    print(
        f"  {level}: {count:,}"
    )

Primary-signature metadata alignment: PASS
Frozen aggregation hierarchy:
  signatures: 362,036
  conditions: 325,190
  timepoints_within_cell: 113,097
  perturbagen_cell_lines: 90,673
  perturbagen_lineages: 55,827
  perturbagens: 7,914


In [35]:
# =============================================================================
# Precompute deterministic aggregation-group mappings
# =============================================================================

def factorize_group_keys(frame, columns):
    keys = pd.MultiIndex.from_frame(
        frame[columns]
    )

    codes, uniques = pd.factorize(
        keys,
        sort=True,
    )

    return (
        codes.astype(np.int32),
        uniques,
    )


condition_id, condition_index = factorize_group_keys(
    primary_signature_metadata,
    condition_keys,
)

timepoint_id, timepoint_index = factorize_group_keys(
    primary_signature_metadata,
    timepoint_keys,
)

cell_line_id, cell_line_index = factorize_group_keys(
    primary_signature_metadata,
    cell_line_keys,
)

lineage_id, lineage_index = factorize_group_keys(
    primary_signature_metadata,
    lineage_keys,
)

perturbagen_id, perturbagen_index = factorize_group_keys(
    primary_signature_metadata,
    ["pert_id"],
)

signature_hierarchy_index = pd.DataFrame(
    {
        "condition_id": condition_id,
        "timepoint_id": timepoint_id,
        "cell_line_id": cell_line_id,
        "lineage_id": lineage_id,
        "perturbagen_id": perturbagen_id,
    }
)


def build_parent_map(
    child_ids,
    parent_ids,
    n_children,
    label,
):
    mapping = (
        pd.DataFrame(
            {
                "child_id": child_ids,
                "parent_id": parent_ids,
            }
        )
        .drop_duplicates()
    )

    if mapping["child_id"].duplicated().any():
        raise RuntimeError(
            f"{label} does not map uniquely "
            "to its parent level."
        )

    mapping = (
        mapping
        .sort_values(
            "child_id",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if not np.array_equal(
        mapping["child_id"].to_numpy(),
        np.arange(
            n_children,
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            f"{label} IDs are not complete and contiguous."
        )

    return mapping[
        "parent_id"
    ].to_numpy(dtype=np.int32)


condition_to_timepoint = build_parent_map(
    condition_id,
    timepoint_id,
    len(condition_index),
    "Condition",
)

timepoint_to_cell_line = build_parent_map(
    timepoint_id,
    cell_line_id,
    len(timepoint_index),
    "Timepoint",
)

cell_line_to_lineage = build_parent_map(
    cell_line_id,
    lineage_id,
    len(cell_line_index),
    "Cell line",
)

lineage_to_perturbagen = build_parent_map(
    lineage_id,
    perturbagen_id,
    len(lineage_index),
    "Lineage",
)

observed_group_counts = {
    "conditions": len(condition_index),
    "timepoints_within_cell": len(timepoint_index),
    "perturbagen_cell_lines": len(cell_line_index),
    "perturbagen_lineages": len(lineage_index),
    "perturbagens": len(perturbagen_index),
}

for level, expected_count in hierarchy_counts.items():
    if level == "signatures":
        continue

    if observed_group_counts[level] != expected_count:
        raise RuntimeError(
            f"Hierarchy count mismatch for {level}: "
            f"{observed_group_counts[level]:,} != "
            f"{expected_count:,}"
        )

print("Deterministic hierarchy mappings: PASS")
print(
    "Signature → condition:",
    f"{len(condition_id):,} → {len(condition_index):,}",
)
print(
    "Condition → timepoint:",
    f"{len(condition_to_timepoint):,} → "
    f"{len(timepoint_index):,}",
)
print(
    "Timepoint → cell line:",
    f"{len(timepoint_to_cell_line):,} → "
    f"{len(cell_line_index):,}",
)
print(
    "Cell line → lineage:",
    f"{len(cell_line_to_lineage):,} → "
    f"{len(lineage_index):,}",
)
print(
    "Lineage → perturbagen:",
    f"{len(lineage_to_perturbagen):,} → "
    f"{len(perturbagen_index):,}",
)

Deterministic hierarchy mappings: PASS
Signature → condition: 362,036 → 325,190
Condition → timepoint: 325,190 → 113,097
Timepoint → cell line: 113,097 → 90,673
Cell line → lineage: 90,673 → 55,827
Lineage → perturbagen: 55,827 → 7,914


In [36]:
# =============================================================================
# Build and validate reusable hierarchical median reductions
# =============================================================================

NUMERICAL_RTOL = 1e-10
NUMERICAL_ATOL = 1e-12


def build_median_reduction_plan(
    parent_ids,
):
    parent_ids = np.asarray(
        parent_ids,
        dtype=np.int64,
    )

    order = np.argsort(
        parent_ids,
        kind="stable",
    )

    sorted_parent_ids = parent_ids[order]

    unique_parents, starts, counts = np.unique(
        sorted_parent_ids,
        return_index=True,
        return_counts=True,
    )

    expected_parents = np.arange(
        len(unique_parents),
        dtype=np.int64,
    )

    if not np.array_equal(
        unique_parents,
        expected_parents,
    ):
        raise RuntimeError(
            "Parent IDs must be complete and contiguous."
        )

    plan = []

    for group_size in np.unique(counts):
        size_mask = counts == group_size

        parent_group_ids = unique_parents[
            size_mask
        ]

        group_starts = starts[
            size_mask
        ]

        child_indices = order[
            group_starts[:, np.newaxis]
            + np.arange(
                group_size,
                dtype=np.int64,
            )
        ]

        plan.append(
            {
                "group_size": int(group_size),
                "parent_ids": (
                    parent_group_ids.astype(
                        np.int32
                    )
                ),
                "child_indices": (
                    child_indices.astype(
                        np.int32
                    )
                ),
            }
        )

    return plan, len(unique_parents)


def apply_median_reduction(
    values,
    plan,
    n_parents,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    input_was_1d = values.ndim == 1

    if input_was_1d:
        values = values[:, np.newaxis]

    if values.ndim != 2:
        raise ValueError(
            "Median reduction expects a 1D or 2D array."
        )

    reduced = np.empty(
        (
            n_parents,
            values.shape[1],
        ),
        dtype=np.float64,
    )

    for block in plan:
        child_values = values[
            block["child_indices"],
            :,
        ]

        reduced[
            block["parent_ids"],
            :,
        ] = np.median(
            child_values,
            axis=1,
        )

    if input_was_1d:
        return reduced[:, 0]

    return reduced


aggregation_plans = {}

aggregation_plans[
    "signature_to_condition"
] = build_median_reduction_plan(
    condition_id
)

aggregation_plans[
    "condition_to_timepoint"
] = build_median_reduction_plan(
    condition_to_timepoint
)

aggregation_plans[
    "timepoint_to_cell_line"
] = build_median_reduction_plan(
    timepoint_to_cell_line
)

aggregation_plans[
    "cell_line_to_lineage"
] = build_median_reduction_plan(
    cell_line_to_lineage
)

aggregation_plans[
    "lineage_to_perturbagen"
] = build_median_reduction_plan(
    lineage_to_perturbagen
)


# Independent small float64 reference validation.
toy_scores = np.array(
    [
        [0.8, -0.3, 0.1],
        [0.2, -0.7, 0.4],
        [-0.4, 0.1, 0.3],
        [0.7, 0.2, -0.5],
        [0.1, 0.6, -0.1],
        [-0.2, 0.4, 0.9],
        [0.5, -0.8, 0.2],
        [0.3, -0.2, 0.6],
        [-0.1, 0.5, 0.7],
    ],
    dtype=np.float64,
)

toy_signature_to_condition = np.array(
    [0, 0, 1, 2, 2, 3, 4, 4, 4],
    dtype=np.int64,
)

toy_condition_to_timepoint = np.array(
    [0, 0, 1, 2, 2],
    dtype=np.int64,
)

toy_timepoint_to_cell = np.array(
    [0, 1, 1],
    dtype=np.int64,
)

toy_cell_to_lineage = np.array(
    [0, 1],
    dtype=np.int64,
)

toy_lineage_to_perturbagen = np.array(
    [0, 0],
    dtype=np.int64,
)


def reference_grouped_median(
    values,
    parent_ids,
):
    value_frame = pd.DataFrame(
        np.asarray(
            values,
            dtype=np.float64,
        )
    )

    value_frame["_parent_id"] = (
        np.asarray(
            parent_ids,
            dtype=np.int64,
        )
    )

    return (
        value_frame
        .groupby(
            "_parent_id",
            sort=True,
        )
        .median()
        .to_numpy(dtype=np.float64)
    )


toy_fast = toy_scores
toy_reference = toy_scores

for toy_parent_ids in (
    toy_signature_to_condition,
    toy_condition_to_timepoint,
    toy_timepoint_to_cell,
    toy_cell_to_lineage,
    toy_lineage_to_perturbagen,
):
    toy_plan, toy_n_parents = (
        build_median_reduction_plan(
            toy_parent_ids
        )
    )

    toy_fast = apply_median_reduction(
        toy_fast,
        toy_plan,
        toy_n_parents,
    )

    toy_reference = reference_grouped_median(
        toy_reference,
        toy_parent_ids,
    )

    np.testing.assert_allclose(
        toy_fast,
        toy_reference,
        rtol=NUMERICAL_RTOL,
        atol=NUMERICAL_ATOL,
    )

print(
    "Hierarchical median reference validation: PASS"
)

for level, (
    plan,
    n_parents,
) in aggregation_plans.items():
    print(
        f"{level}: "
        f"{n_parents:,} groups; "
        f"{len(plan)} distinct group sizes; "
        f"max size = "
        f"{max(block['group_size'] for block in plan):,}"
    )

Hierarchical median reference validation: PASS
signature_to_condition: 325,190 groups; 74 distinct group sizes; max size = 395
condition_to_timepoint: 113,097 groups; 36 distinct group sizes; max size = 41
timepoint_to_cell_line: 90,673 groups; 5 distinct group sizes; max size = 5
cell_line_to_lineage: 55,827 groups; 18 distinct group sizes; max size = 19
lineage_to_perturbagen: 7,914 groups; 17 distinct group sizes; max size = 20


In [37]:
# =============================================================================
# Define and validate the complete frozen aggregation path
# =============================================================================

def aggregate_signature_scores(
    signature_scores,
):
    signature_scores = np.asarray(
        signature_scores,
        dtype=np.float64,
    )

    if signature_scores.ndim == 1:
        signature_scores = (
            signature_scores[:, np.newaxis]
        )

    if (
        signature_scores.ndim != 2
        or signature_scores.shape[0]
        != hierarchy_counts["signatures"]
    ):
        raise ValueError(
            "Signature scores do not match the frozen "
            "primary signature universe."
        )

    condition_scores = apply_median_reduction(
        signature_scores,
        *aggregation_plans[
            "signature_to_condition"
        ],
    )

    timepoint_scores = apply_median_reduction(
        condition_scores,
        *aggregation_plans[
            "condition_to_timepoint"
        ],
    )

    cell_line_scores = apply_median_reduction(
        timepoint_scores,
        *aggregation_plans[
            "timepoint_to_cell_line"
        ],
    )

    lineage_scores = apply_median_reduction(
        cell_line_scores,
        *aggregation_plans[
            "cell_line_to_lineage"
        ],
    )

    perturbagen_scores = apply_median_reduction(
        lineage_scores,
        *aggregation_plans[
            "lineage_to_perturbagen"
        ],
    )

    return {
        "condition": condition_scores,
        "timepoint": timepoint_scores,
        "cell_line": cell_line_scores,
        "lineage": lineage_scores,
        "perturbagen": perturbagen_scores,
    }


# Outcome-blind synthetic scores for structural validation only.
synthetic_signature_scores = np.column_stack(
    [
        np.sin(
            np.arange(
                hierarchy_counts["signatures"],
                dtype=np.float64,
            )
            / scale
        )
        for scale in (
            101.0,
            211.0,
            307.0,
        )
    ]
)

synthetic_aggregation = (
    aggregate_signature_scores(
        synthetic_signature_scores
    )
)

expected_shapes = {
    "condition": (
        hierarchy_counts["conditions"],
        3,
    ),
    "timepoint": (
        hierarchy_counts[
            "timepoints_within_cell"
        ],
        3,
    ),
    "cell_line": (
        hierarchy_counts[
            "perturbagen_cell_lines"
        ],
        3,
    ),
    "lineage": (
        hierarchy_counts[
            "perturbagen_lineages"
        ],
        3,
    ),
    "perturbagen": (
        hierarchy_counts["perturbagens"],
        3,
    ),
}

for level, expected_shape in (
    expected_shapes.items()
):
    observed = synthetic_aggregation[level]

    if observed.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected {level} score shape: "
            f"{observed.shape} != "
            f"{expected_shape}"
        )

    if not np.isfinite(observed).all():
        raise RuntimeError(
            f"Non-finite synthetic {level} scores."
        )

print(
    "Complete frozen aggregation path: PASS"
)

for level in (
    "condition",
    "timepoint",
    "cell_line",
    "lineage",
    "perturbagen",
):
    print(
        f"{level}: "
        f"{synthetic_aggregation[level].shape}"
    )

Complete frozen aggregation path: PASS
condition: (325190, 3)
timepoint: (113097, 3)
cell_line: (90673, 3)
lineage: (55827, 3)
perturbagen: (7914, 3)


In [38]:
# =============================================================================
# Prepare frozen query arrays for pseudo-program cosine scoring
# =============================================================================

bing_row_lookup = (
    bing_gene_universe
    .reset_index(
        names="bing_matrix_row"
    )
    .set_index("gene_id")[
        "bing_matrix_row"
    ]
)

query_gene_ids = (
    coupled_query_weights[
        "gene_id"
    ]
    .to_numpy(dtype=np.int64)
)

query_bing_row_indices = (
    coupled_query_weights[
        "gene_id"
    ]
    .map(bing_row_lookup)
)

if query_bing_row_indices.isna().any():
    raise RuntimeError(
        "At least one frozen query gene is missing "
        "from the primary BING matrix."
    )

query_bing_row_indices = (
    query_bing_row_indices
    .to_numpy(dtype=np.int64)
)

frozen_weight_matrix = (
    coupled_query_weights[
        list(PROGRAM_IDS)
    ]
    .to_numpy(dtype=np.float64)
)

resistance_like_multiplier_vector = (
    program_source_metadata
    .set_index("program_id")
    .loc[
        list(PROGRAM_IDS),
        "resistance_like_orientation_multiplier",
    ]
    .to_numpy(dtype=np.float64)
)

frozen_weight_norms = np.linalg.norm(
    frozen_weight_matrix,
    axis=0,
)

if (
    len(np.unique(query_gene_ids))
    != 1495
):
    raise RuntimeError(
        "Frozen query support is not 1,495 unique genes."
    )

if not np.isfinite(
    frozen_weight_matrix
).all():
    raise RuntimeError(
        "Frozen query weights contain non-finite values."
    )

if not np.all(
    frozen_weight_norms > 0
):
    raise RuntimeError(
        "At least one frozen query has zero weight norm."
    )

expected_weight_norms = (
    query_norms
    .reindex(PROGRAM_IDS)
    .to_numpy(dtype=np.float64)
)

np.testing.assert_allclose(
    frozen_weight_norms,
    expected_weight_norms,
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

print(
    "Shared frozen query support:",
    f"{len(query_gene_ids):,}",
    "genes",
)
print(
    "Weight matrix shape:",
    frozen_weight_matrix.shape,
)
print(
    "Primary BING matrix row range:",
    f"{query_bing_row_indices.min():,}"
    " to "
    f"{query_bing_row_indices.max():,}",
)
print(
    "Frozen weight norms:",
    dict(
        zip(
            PROGRAM_IDS,
            frozen_weight_norms,
        )
    ),
)
print(
    "Resistance-like multipliers:",
    dict(
        zip(
            PROGRAM_IDS,
            resistance_like_multiplier_vector.astype(int),
        )
    ),
)

Shared frozen query support: 1,495 genes
Weight matrix shape: (1495, 3)
Primary BING matrix row range: 2 to 10,149
Frozen weight norms: {'CONSENSUS_TX_01': np.float64(0.8507412531758485), 'CONSENSUS_TX_02': np.float64(0.8016572205527611), 'CONSENSUS_TX_03': np.float64(0.8130261461852406)}
Resistance-like multipliers: {'CONSENSUS_TX_01': np.int64(-1), 'CONSENSUS_TX_02': np.int64(1), 'CONSENSUS_TX_03': np.int64(1)}


In [39]:
# =============================================================================
# Generate coupled outcome-blind benchmark pseudo-program weights
# =============================================================================

query_stratum_row_indices = {
    int(stratum_id): (
        np.flatnonzero(
            coupled_query_weights[
                "stratum_id"
            ].to_numpy(dtype=np.int64)
            == int(stratum_id)
        )
    )
    for stratum_id in STRATUM_IDS
}


def generate_coupled_pseudo_weights(
    replicate_id,
    namespace=RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
):
    pseudo_weights = (
        frozen_weight_matrix.copy()
    )

    for stratum_id in STRATUM_IDS:
        row_indices = (
            query_stratum_row_indices[
                stratum_id
            ]
        )

        permutation = (
            fisher_yates_permutation(
                n_positions=len(row_indices),
                namespace=namespace,
                replicate_id=replicate_id,
                stratum_id=stratum_id,
            )
        )

        pseudo_weights[
            row_indices,
            :,
        ] = frozen_weight_matrix[
            row_indices[permutation],
            :,
        ]

    return pseudo_weights


benchmark_pseudo_weights_0 = (
    generate_coupled_pseudo_weights(
        replicate_id=0
    )
)

benchmark_pseudo_weights_0_repeat = (
    generate_coupled_pseudo_weights(
        replicate_id=0
    )
)

if not np.array_equal(
    benchmark_pseudo_weights_0,
    benchmark_pseudo_weights_0_repeat,
):
    raise RuntimeError(
        "Benchmark pseudo-program replicate "
        "is not reproducible."
    )

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    np.testing.assert_array_equal(
        np.sort(
            benchmark_pseudo_weights_0[
                :,
                program_index,
            ]
        ),
        np.sort(
            frozen_weight_matrix[
                :,
                program_index,
            ]
        ),
    )

pseudo_weight_norms = np.linalg.norm(
    benchmark_pseudo_weights_0,
    axis=0,
)

np.testing.assert_allclose(
    pseudo_weight_norms,
    frozen_weight_norms,
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if np.array_equal(
    benchmark_pseudo_weights_0,
    frozen_weight_matrix,
):
    raise RuntimeError(
        "Benchmark replicate 0 unexpectedly produced "
        "the global identity assignment."
    )

print(
    "Benchmark pseudo-program generation: PASS"
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print("Replicate:", 0)
print(
    "Coupled pseudo-weight matrix shape:",
    benchmark_pseudo_weights_0.shape,
)
print(
    "Weight norms preserved:",
    dict(
        zip(
            PROGRAM_IDS,
            pseudo_weight_norms,
        )
    ),
)

Benchmark pseudo-program generation: PASS
Namespace: 3
Replicate: 0
Coupled pseudo-weight matrix shape: (1495, 3)
Weight norms preserved: {'CONSENSUS_TX_01': np.float64(0.8507412531758485), 'CONSENSUS_TX_02': np.float64(0.8016572205527613), 'CONSENSUS_TX_03': np.float64(0.8130261461852408)}


In [40]:
# =============================================================================
# Define and validate the float64 pseudo-program cosine kernel
# =============================================================================

def score_pseudo_program_block(
    signature_block,
    weight_matrix,
):
    signature_block = np.asarray(
        signature_block,
        dtype=np.float64,
    )

    weight_matrix = np.asarray(
        weight_matrix,
        dtype=np.float64,
    )

    if signature_block.ndim != 2:
        raise ValueError(
            "Signature block must be genes × signatures."
        )

    if weight_matrix.shape != (
        signature_block.shape[0],
        len(PROGRAM_IDS),
    ):
        raise ValueError(
            "Weight matrix and signature block "
            "have incompatible shapes."
        )

    if not np.isfinite(signature_block).all():
        raise ValueError(
            "Signature block contains non-finite values."
        )

    signature_norms = np.linalg.norm(
        signature_block,
        axis=0,
    )

    weight_norms = np.linalg.norm(
        weight_matrix,
        axis=0,
    )

    numerators = (
        weight_matrix.T
        @ signature_block
    )

    denominators = (
        weight_norms[:, np.newaxis]
        * signature_norms[np.newaxis, :]
    )

    scores = np.full(
        numerators.shape,
        np.nan,
        dtype=np.float64,
    )

    evaluable = signature_norms > 0

    scores[
        :,
        evaluable,
    ] = (
        numerators[:, evaluable]
        / denominators[:, evaluable]
    )

    return scores.T, evaluable


# Independent small reference calculation.
toy_signature_block = np.array(
    [
        [0.5, -0.2, 0.7, 0.0],
        [-0.3, 0.4, 0.1, 0.0],
        [0.8, 0.6, -0.5, 0.0],
        [0.2, -0.7, 0.3, 0.0],
    ],
    dtype=np.float64,
)

toy_weight_matrix = np.array(
    [
        [0.4, -0.2, 0.1],
        [-0.1, 0.5, 0.3],
        [0.7, 0.2, -0.4],
        [-0.3, 0.1, 0.6],
    ],
    dtype=np.float64,
)

toy_scores, toy_evaluable = (
    score_pseudo_program_block(
        toy_signature_block,
        toy_weight_matrix,
    )
)

toy_reference = np.full(
    toy_scores.shape,
    np.nan,
    dtype=np.float64,
)

for signature_index in range(
    toy_signature_block.shape[1]
):
    z = toy_signature_block[
        :,
        signature_index,
    ]

    z_norm = np.sqrt(
        np.sum(z * z)
    )

    if z_norm == 0:
        continue

    for program_index in range(
        toy_weight_matrix.shape[1]
    ):
        w = toy_weight_matrix[
            :,
            program_index,
        ]

        toy_reference[
            signature_index,
            program_index,
        ] = (
            np.sum(w * z)
            / (
                np.sqrt(np.sum(w * w))
                * z_norm
            )
        )

np.testing.assert_allclose(
    toy_scores[
        toy_evaluable,
        :,
    ],
    toy_reference[
        toy_evaluable,
        :,
    ],
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if not np.array_equal(
    toy_evaluable,
    np.array(
        [True, True, True, False]
    ),
):
    raise RuntimeError(
        "Zero-norm signature evaluability "
        "was not handled as specified."
    )

if np.nanmax(
    np.abs(toy_scores)
) > 1 + NUMERICAL_ATOL:
    raise RuntimeError(
        "Cosine score exceeded its theoretical range."
    )

print(
    "Float64 cosine kernel reference validation: PASS"
)
print(
    "Toy evaluable signatures:",
    int(toy_evaluable.sum()),
    "/",
    len(toy_evaluable),
)
print(
    "Zero-norm signatures marked non-evaluable:",
    int((~toy_evaluable).sum()),
)

Float64 cosine kernel reference validation: PASS
Toy evaluable signatures: 3 / 4
Zero-norm signatures marked non-evaluable: 1


In [41]:
# =============================================================================
# Define the outcome-blind benchmark scoring path
# =============================================================================

BENCHMARK_SIGNATURE_BLOCK_SIZE = 4096


def score_benchmark_replicate(
    replicate_id,
):
    pseudo_weights = (
        generate_coupled_pseudo_weights(
            replicate_id=replicate_id,
            namespace=RNG_NAMESPACES[
                "outcome_blind_benchmark"
            ],
        )
    )

    n_signatures = hierarchy_counts[
        "signatures"
    ]

    signature_scores = np.empty(
        (
            n_signatures,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    signature_evaluable = np.empty(
        n_signatures,
        dtype=bool,
    )

    for block_start in range(
        0,
        n_signatures,
        BENCHMARK_SIGNATURE_BLOCK_SIZE,
    ):
        block_stop = min(
            block_start
            + BENCHMARK_SIGNATURE_BLOCK_SIZE,
            n_signatures,
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                query_bing_row_indices,
                block_start:block_stop,
            ],
            dtype=np.float64,
        )

        block_scores, block_evaluable = (
            score_pseudo_program_block(
                signature_block,
                pseudo_weights,
            )
        )

        signature_scores[
            block_start:block_stop,
            :,
        ] = block_scores

        signature_evaluable[
            block_start:block_stop
        ] = block_evaluable

    if not signature_evaluable.all():
        raise RuntimeError(
            "Outcome-blind benchmark encountered "
            "non-evaluable primary signatures."
        )

    raw_aggregation = aggregate_signature_scores(
        signature_scores
    )

    resistance_like_aggregation = {
        level: (
            scores
            * resistance_like_multiplier_vector[
                np.newaxis,
                :
            ]
        )
        for level, scores
        in raw_aggregation.items()
    }

    return {
        "replicate_id": int(replicate_id),
        "raw": raw_aggregation,
        "resistance_like": (
            resistance_like_aggregation
        ),
    }


print(
    "Outcome-blind benchmark scoring path: DEFINED"
)
print(
    "Signature block size:",
    BENCHMARK_SIGNATURE_BLOCK_SIZE,
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print(
    "Full primary signatures per replicate:",
    f"{hierarchy_counts['signatures']:,}",
)
print(
    "No full benchmark replicate executed: YES"
)

Outcome-blind benchmark scoring path: DEFINED
Signature block size: 4096
Namespace: 3
Full primary signatures per replicate: 362,036
No full benchmark replicate executed: YES


In [42]:
# =============================================================================
# Smoke-test pseudo-program scoring on small primary-signature blocks
# =============================================================================

SMOKE_TEST_BLOCK_STARTS = (
    0,
    hierarchy_counts["signatures"]
    - BENCHMARK_SIGNATURE_BLOCK_SIZE,
)

smoke_pseudo_weights = (
    generate_coupled_pseudo_weights(
        replicate_id=0,
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
    )
)

smoke_test_records = []

for block_start in SMOKE_TEST_BLOCK_STARTS:
    block_stop = min(
        block_start
        + BENCHMARK_SIGNATURE_BLOCK_SIZE,
        hierarchy_counts["signatures"],
    )

    signature_block = np.asarray(
        primary_bing_matrix[
            query_bing_row_indices,
            block_start:block_stop,
        ],
        dtype=np.float64,
    )

    block_scores, block_evaluable = (
        score_pseudo_program_block(
            signature_block,
            smoke_pseudo_weights,
        )
    )

    if not block_evaluable.all():
        raise RuntimeError(
            "Smoke test encountered a non-evaluable "
            "primary signature."
        )

    if not np.isfinite(block_scores).all():
        raise RuntimeError(
            "Smoke test produced non-finite cosine scores."
        )

    if (
        np.max(np.abs(block_scores))
        > 1 + NUMERICAL_ATOL
    ):
        raise RuntimeError(
            "Smoke-test cosine score exceeded "
            "the theoretical range."
        )

    smoke_test_records.append(
        {
            "block_start": int(block_start),
            "block_stop": int(block_stop),
            "signatures": int(
                block_stop - block_start
            ),
            "score_min": float(
                block_scores.min()
            ),
            "score_max": float(
                block_scores.max()
            ),
        }
    )

smoke_test_summary = pd.DataFrame(
    smoke_test_records
)

print("Pseudo-program scoring smoke test: PASS")
print(
    "Signatures tested:",
    f"{smoke_test_summary['signatures'].sum():,}",
)
print(
    "Full benchmark replicate executed: NO"
)

display(smoke_test_summary)

Pseudo-program scoring smoke test: PASS
Signatures tested: 8,192
Full benchmark replicate executed: NO


,block_start,block_stop,signatures,score_min,score_max
0,0,4096,4096,-0.101870,0.106608
1,357940,362036,4096,-0.106607,0.104057
